In [ ]:
import requests
import pandas as pd

API_KEY = "9xwByEiAgdxN6lDLtrytLNPRMLfThevoqzPwcAo0"  # thay bằng key EIA của bạn
BASE_URL = "https://api.eia.gov/v2/electricity/rto/daily-region-data/data/"

def fetch_eia_data(start, end, respondent, tz, types):
    """Lấy dữ liệu từ EIA API"""
    all_frames = []
    for t in types:
        params = {
            "api_key": API_KEY,
            "start": start,
            "end": end,
            "frequency": "daily",
            "data[0]": "value",
            "facets[respondent][]": respondent,
            "facets[timezone][]": tz,
            "facets[type][]": t,   # D = Demand, NG = Net generation, TI = Interchange
            "sort[0][column]": "period",
            "sort[0][direction]": "asc",
            "length": 5000
        }
        r = requests.get(BASE_URL, params=params)
        r.raise_for_status()
        data = r.json()["response"]["data"]
        df = pd.DataFrame(data)
        df["type"] = t
        all_frames.append(df)
    return pd.concat(all_frames, ignore_index=True)

# === Lấy dữ liệu CAL, Pacific, 2015-01-01 → 2025-08-31 ===
df = fetch_eia_data(
    start="2015-01-01",
    end="2025-11-12",
    respondent="CAL",
    tz="Pacific",
    types=["D", "NG", "TI"]
)

# Xử lý dữ liệu: chỉ lấy 5 cột cần
df = df[["period", "timezone", "type", "value"]]

# Pivot để tách type thành các cột demand, net_generation, interchange
df_pivot = df.pivot_table(
    index=["period", "timezone"],
    columns="type",
    values="value",
    aggfunc="first"
).reset_index()

# Đổi tên cột cho dễ hiểu
df_pivot = df_pivot.rename(columns={
    "period": "date",
    "timezone": "location",
    "D": "demand",
    "NG": "net_generation",
    "TI": "interchange"
})

# Chuyển giá trị sang số và làm tròn 2 chữ số thập phân
for col in ["demand", "net_generation", "interchange"]:
    df_pivot[col] = pd.to_numeric(df_pivot[col], errors="coerce").round(2)

print(df_pivot.count())
df_pivot.to_csv("elec9.csv", index=False)


type
date              2506
location          2506
demand            2506
net_generation    2506
interchange       2501
dtype: int64


In [23]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [24]:
# ================================
# CELL 2: Configuration và API setup
# ================================

# API Configuration
API_TOKEN = "oOzXUWxodCEvUJjbYLpMKCxWIArJrUpM"
headers = {"token": API_TOKEN}
base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2"

# Date range
START_DATE, END_DATE = "2019-01-01", "2025-11-12"

# Đại diện 5 thành phố đông dân nhất California
stations = {
    "GHCND:USW00023174": "Los Angeles LAX",
    "GHCND:USW00023188": "San Diego",
    "GHCND:USW00023234": "San Francisco",
    "GHCND:USW00023232": "Sacramento",
    "GHCND:USW00093193": "Fresno"
}

# Các datatype cần lấy
datatypes = ["TMAX", "TMIN", "TAVG", "PRCP", "RHAV"]



In [25]:
def get_station_data(station_id, start_date, end_date, retries=3):
    url = f"{base_url}/data"
    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt   = datetime.strptime(end_date, "%Y-%m-%d")
    all_data = []

    while start_dt < end_dt:
        current_end = min(start_dt + timedelta(days=180), end_dt)
        params = {
            "datasetid": "GHCND",
            "stationid": station_id,
            "startdate": start_dt.strftime("%Y-%m-%d"),
            "enddate": current_end.strftime("%Y-%m-%d"),
            "datatypeid": ",".join(datatypes),
            "limit": 1000,
            "units": "metric"
        }
        for attempt in range(retries):
            try:
                resp = requests.get(url, headers=headers, params=params)
                if resp.status_code == 200 and "results" in resp.json():
                    all_data.extend(resp.json()["results"])
                    break
                elif resp.status_code == 429:  # Rate limit
                    time.sleep(60)
                else:
                    break
            except:
                time.sleep(5)
        start_dt = current_end + timedelta(days=1)
        time.sleep(1)
    return pd.DataFrame(all_data) if all_data else None


In [26]:
def process_station_data(df, station_name):
    if df is None or df.empty:
        return pd.DataFrame()
    pivot_df = df.pivot_table(
        index='date', columns='datatype', values='value', aggfunc='first'
    ).reset_index()
    pivot_df['date'] = pd.to_datetime(pivot_df['date'])
    pivot_df['station'] = station_name
    
    return pivot_df


In [27]:
all_data = {}
for sid, name in stations.items():
    raw = get_station_data(sid, START_DATE, END_DATE)
    proc = process_station_data(raw, name)
    if not proc.empty:
        all_data[sid] = proc


In [28]:
# ================================
# CELL 8 & 9: Làm tròn và lưu dữ liệu
# ================================

def round_weather_data(df, decimal_places=None):
    if df.empty:
        return df

    rounded_df = df.copy()
    
    if decimal_places is None:
        decimal_places = {
            'TMAX': 1, 'TMIN': 1, 'TAVG': 1, 'TEMP_RANGE': 1,
            'CDD': 1, 'HDD': 1, 'PRCP': 1,
            'RHAV': 0, 'AWND': 1,
        }
    
    for col, decimals in decimal_places.items():
        if col in rounded_df.columns:
            rounded_df[col] = rounded_df[col].round(decimals)
            if decimals == 0 and not rounded_df[col].isna().any():
                rounded_df[col] = rounded_df[col].astype(int)
    
    return rounded_df



In [29]:
# ================================
# CELL A: Lấy dữ liệu gió (Wind Speed)
# ================================
def get_wind_data(station_id, start_date, end_date, retries=3):
    url = f"{base_url}/data"
    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt   = datetime.strptime(end_date, "%Y-%m-%d")
    wind_data = []

    while start_dt < end_dt:
        current_end = min(start_dt + timedelta(days=180), end_dt)
        params = {
            "datasetid": "GHCND",
            "stationid": station_id,
            "startdate": start_dt.strftime("%Y-%m-%d"),
            "enddate": current_end.strftime("%Y-%m-%d"),
            "datatypeid": "AWND",  # chỉ lấy gió
            "limit": 1000,
            "units": "metric"
        }
        for attempt in range(retries):
            try:
                resp = requests.get(url, headers=headers, params=params)
                if resp.status_code == 200 and "results" in resp.json():
                    wind_data.extend(resp.json()["results"])
                    break
                elif resp.status_code == 429:  # Rate limit
                    time.sleep(60)
                else:
                    break
            except:
                time.sleep(5)
        start_dt = current_end + timedelta(days=1)
        time.sleep(1)
    return pd.DataFrame(wind_data) if wind_data else None


In [30]:
# ================================
# CELL B: Xử lý dữ liệu gió - ..........................................................................................
# ================================
def process_wind_data(df, station_name):
    if df is None or df.empty:
        return pd.DataFrame()
    wind_df = df.pivot_table(
        index='date', columns='datatype', values='value', aggfunc='first'
    ).reset_index()
    wind_df['date'] = pd.to_datetime(wind_df['date'])
    wind_df['station'] = station_name


    return wind_df


In [31]:
# ================================
# CELL C: Lấy dữ liệu gió cho tất cả các trạm
# ================================
wind_all_data = {}
for sid, name in stations.items():
    raw_wind = get_wind_data(sid, START_DATE, END_DATE)
    proc_wind = process_wind_data(raw_wind, name)
    if not proc_wind.empty:
        wind_all_data[sid] = proc_wind


In [32]:
# CELL E: Merge dữ liệu gió vào weather data
combined = pd.concat(all_data.values(), ignore_index=True)
wind_combined = pd.concat(wind_all_data.values(), ignore_index=True)
daily_weather_full = combined.merge(wind_combined, on = ['date', 'station'], how='left')
print(daily_weather_full.head())


datatype       date  PRCP  RHAV  TAVG  TMAX  TMIN          station  AWND
0        2019-01-01   0.0  18.0  12.1  16.7   5.0  Los Angeles LAX   2.8
1        2019-01-02   0.0  31.0  11.2  16.1   3.9  Los Angeles LAX   2.1
2        2019-01-03   0.0  44.0  12.4  18.9   7.2  Los Angeles LAX   2.8
3        2019-01-04   0.0  52.0  12.4  17.2   6.7  Los Angeles LAX   2.1
4        2019-01-05  14.2  74.0  12.5  14.4   8.9  Los Angeles LAX   3.4


In [33]:
print(daily_weather_full.isna().sum())

datatype
date          0
PRCP          3
RHAV       1579
TAVG        335
TMAX          1
TMIN          2
station       0
AWND        367
dtype: int64


In [34]:
from sklearn.ensemble import RandomForestRegressor
def fill_humidity_with_rf(df):
    data = df[['RHAV','TAVG','PRCP']].copy()
    train = data.dropna(subset=['RHAV'])
    test  = data[data['RHAV'].isna()]

    if not test.empty:
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(train[['TAVG','PRCP']], train['RHAV'])
        df.loc[df['RHAV'].isna(), 'RHAV'] = model.predict(test[['TAVG','PRCP']])
    return df

In [35]:


def clean_weather_data(df):
    df = df.copy()
    
    # --- 1. Nhiệt độ (TMAX, TMIN, TAVG) ---
    for col in ['TMAX', 'TMIN']:
        if col in df.columns:
            df[col] = df[col].interpolate(method='linear')
    if 'TAVG' not in df or df['TAVG'].isna().any():
        if 'TMAX' in df and 'TMIN' in df:
            df['TAVG'] = df['TAVG'].fillna((df['TMAX']+df['TMIN'])/2)
    # CDD & HDD
    if 'TAVG' in df:
        df['CDD'] = np.maximum(df['TAVG'] - 18.3, 0)
        df['HDD'] = np.maximum(18.3 - df['TAVG'], 0)


    # --- 3. Tốc độ gió (WIND_AVG) ---
    if 'AWND' in df.columns:
        df['AWND'] = df['AWND'].interpolate(method='linear')


    # --- 5. Lượng mưa (PRCP) ---
    if 'PRCP' in df.columns:
        df['PRCP'] = df['PRCP'].fillna(0)  # giả định thiếu = không mưa

    return df


In [36]:
# ================================
# CELL F: Lưu ra csv


print("✅ Merged weather + wind sample:")
print(daily_weather_full.head())
# Thực hiện làm tròn và lưu
if not daily_weather_full.empty:
    final_weather = fill_humidity_with_rf(daily_weather_full)
    final_weather = clean_weather_data(daily_weather_full)

    rounded_weather = round_weather_data(final_weather)

    # Lưu ra CSV
    rounded_weather.to_csv("fake_full_data.csv", index=False)
   


✅ Merged weather + wind sample:
datatype       date  PRCP  RHAV  TAVG  TMAX  TMIN          station  AWND
0        2019-01-01   0.0  18.0  12.1  16.7   5.0  Los Angeles LAX   2.8
1        2019-01-02   0.0  31.0  11.2  16.1   3.9  Los Angeles LAX   2.1
2        2019-01-03   0.0  44.0  12.4  18.9   7.2  Los Angeles LAX   2.8
3        2019-01-04   0.0  52.0  12.4  17.2   6.7  Los Angeles LAX   2.1
4        2019-01-05  14.2  74.0  12.5  14.4   8.9  Los Angeles LAX   3.4


In [37]:
print(rounded_weather.isna().sum())

datatype
date       0
PRCP       0
RHAV       0
TAVG       0
TMAX       0
TMIN       0
station    0
AWND       0
CDD        0
HDD        0
dtype: int64


In [38]:
import pandas as pd


# Khoảng ngày đầy đủ
full_range = pd.date_range(start="2019-01-01", end="2025-09-01", freq="D")

missing_by_station = {}

# Lặp qua từng trạm
for station, group in rounded_weather.groupby("station"):
    existing_days = pd.DatetimeIndex(group['date'].unique())
    missing_days = full_range.difference(existing_days)
    
    missing_by_station[station] = missing_days
    print(f"🏷️ {station}: Thiếu {len(missing_days)} ngày")

# Ví dụ: in 10 ngày thiếu đầu tiên của 1 trạm
sample_station = list(missing_by_station.keys())[0]
print(f"\n📅 Sample missing dates for {sample_station}:")
print(missing_by_station[sample_station][:10])
# Gộp lại thành DataFrame để lưu
missing_records = []
for station, days in missing_by_station.items():
    for d in days:
        missing_records.append({"station": station, "missing_date": d})

missing_df = pd.DataFrame(missing_records)
missing_df.to_csv("missing_dates_by_station.csv", index=False)

print("✅ Đã lưu danh sách ngày thiếu theo trạm vào missing_dates_by_station.csv")


🏷️ Fresno: Thiếu 0 ngày
🏷️ Los Angeles LAX: Thiếu 0 ngày
🏷️ Sacramento: Thiếu 181 ngày
🏷️ San Diego: Thiếu 0 ngày
🏷️ San Francisco: Thiếu 0 ngày

📅 Sample missing dates for Fresno:
DatetimeIndex([], dtype='datetime64[ns]', freq='D')
✅ Đã lưu danh sách ngày thiếu theo trạm vào missing_dates_by_station.csv


In [39]:
import pandas as pd
weather_df = pd.read_csv("full_data_weather2.csv")
weather_df.isnull().sum()

date       0
PRCP       0
RHAV       0
TAVG       0
TMAX       0
TMIN       0
station    0
AWND       0
CDD        0
HDD        0
dtype: int64

Thử xem mối quan hệ giữa nhiệt độ trung bình và lượng điện tiêu thụ

In [40]:
# import the visualization package: seaborn
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 

In [41]:

# Đọc 2 file csv
weather_df = pd.read_csv("full_data_weather2.csv", parse_dates=["date"])
electric_df = pd.read_csv("elec9.csv", parse_dates=["date"])

# Tính trung bình dữ liệu thời tiết theo ngày (vì có nhiều trạm)
weather_daily = (
    weather_df
    .groupby("date")
    .agg({
        "PRCP": "mean",
        "RHAV": "mean",
        "TAVG": "mean",
        "TMAX": "mean",
        "TMIN": "mean",
        "AWND": "mean",
        "CDD": "mean",
        "HDD": "mean"
    })
    .reset_index()
)

# Merge 2 bảng theo ngày
merged_df = pd.merge(electric_df, weather_daily, on="date", how="inner")

# Xem kết quả
print(merged_df.head())


        date location  demand  net_generation  interchange  PRCP  RHAV   TAVG  \
0 2019-01-01  Pacific  646739          465625    -148563.0  0.00  43.8   9.28   
1 2019-01-02  Pacific  713041          488700    -195461.0  0.00  54.4   8.04   
2 2019-01-03  Pacific  723967          484096    -216036.0  0.00  58.8   8.56   
3 2019-01-04  Pacific  715678          477365    -218420.0  0.00  62.6   9.00   
4 2019-01-05  Pacific  706611          451040    -226354.0  7.24  74.2  10.46   

    TMAX  TMIN  AWND  CDD    HDD  
0  13.68  4.46  3.16  0.0   9.02  
1  13.90  2.24  1.12  0.0  10.26  
2  15.12  3.36  1.44  0.0   9.74  
3  16.54  3.56  1.26  0.0   9.30  
4  13.98  6.76  4.12  0.0   7.84  


In [42]:
width = 12
height = 10
plt.figure(figsize=(width, height))
sns.regplot(x="TAVG", y="demand", data=merged_df)
plt.ylim(0,)

(0.0, 1224013.0)